In [1]:
import sqlite3
from datetime import datetime, timedelta

In [5]:
db_path = "FVEDB.db"          # path to your SQLite file
table_name = "MqttData"     # table name
timestamp_column = "DateTime"  # timestamp column
power_column = "P_GRID"          # power column
day = "2026-03-25"              # day to calculate

In [8]:
def parse_timestamp(ts):
    ts = str(ts).strip()
    for fmt in (
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%dT%H:%M:%S",
    ):
        try:
            return datetime.strptime(ts, fmt)
        except ValueError:
            pass
    raise ValueError(f"Unsupported timestamp format: {ts}")


def fetch_day_data(conn, table_name, timestamp_column, power_column, day):
    start_dt = datetime.strptime(day, "%Y-%m-%d")
    end_dt = start_dt + timedelta(days=1)

    query = f"""
        SELECT {timestamp_column}, {power_column}
        FROM {table_name}
        WHERE {timestamp_column} >= ?
          AND {timestamp_column} < ?
        ORDER BY {timestamp_column} ASC
    """

    rows = conn.execute(
        query,
        (
            start_dt.strftime("%Y-%m-%d %H:%M:%S"),
            end_dt.strftime("%Y-%m-%d %H:%M:%S"),
        ),
    ).fetchall()

    return [(parse_timestamp(ts), float(power)) for ts, power in rows]


def calculate_energy_for_day(samples):
    """
    Energy = sum(delta_time_seconds * positive_previous_power)

    Negative power values are ignored (treated as 0).

    Returns:
        energy_ws  = watt-seconds
        energy_wh  = watt-hours
        energy_kwh = kilowatt-hours
    """
    if len(samples) < 2:
        return 0.0, 0.0, 0.0

    total_ws = 0.0
    prev_ts, prev_power = samples[0]

    for current_ts, current_power in samples[1:]:
        delta_seconds = (current_ts - prev_ts).total_seconds()

        if delta_seconds < 0:
            raise ValueError("Timestamps are not sorted correctly.")

        positive_power = max(-prev_power, 0.0)
        total_ws += delta_seconds * positive_power

        prev_ts, prev_power = current_ts, current_power

    total_wh = total_ws / 3600.0
    total_kwh = total_wh / 1000.0

    return total_ws, total_wh, total_kwh

In [9]:
conn = sqlite3.connect(db_path)

samples = fetch_day_data(
    conn,
    table_name=table_name,
    timestamp_column=timestamp_column,
    power_column=power_column,
    day=day,
)

energy_ws, energy_wh, energy_kwh = calculate_energy_for_day(samples)

print(f"Day: {day}")
print(f"Number of samples: {len(samples)}")
print(f"Energy: {energy_ws:.2f} W·s")
print(f"Energy: {energy_wh:.4f} Wh")
print(f"Energy: {energy_kwh:.6f} kWh")

conn.close()

Day: 2026-03-25
Number of samples: 2845
Energy: 127867096.00 W·s
Energy: 35518.6378 Wh
Energy: 35.518638 kWh
